# HydroTwin Exploratory Data Analysis

This notebook explores the cleaned SCADA data for the Turbine Guide Bearing and Upper Guide Bearing.

The analysis focuses on:

1. Operating variables
2. Bearing vibration behaviour
3. Bearing temperature behaviour
4. Relationships between TGB and UGB
5. Relationships between bearing measurements and unit operating conditions

The results will support feature understanding and model development for Linear Regression, LSTM, and XGBoost.

## 1. Import Libraries

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 2. Project Paths

The cleaned dataset generated during the data preparation stage is loaded from the HydroTwin prepared data directory.

In [ ]:
PROJECT_ROOT = Path("..").resolve()

PREPARED_DIR = PROJECT_ROOT / "data" / "prepared"

CLEANED_FILE = (
    PREPARED_DIR /
    "hydrotwin_cleaned_data.csv"
)

print("Project root:")
print(PROJECT_ROOT)

print("\nCleaned dataset:")
print(CLEANED_FILE)

print(
    "\nFile exists:",
    CLEANED_FILE.exists()
)

## 3. Load Cleaned Dataset

In [ ]:
data = pd.read_csv(
    CLEANED_FILE
)

data["timestamp"] = pd.to_datetime(
    data["timestamp"]
)

print(
    "Dataset shape:",
    data.shape
)

data.head()

In [ ]:
print(
    "Columns:"
)

for column in data.columns:
    print(column)

## 4. Dataset Overview

The cleaned dataset is summarized to confirm its size, time range, and remaining data quality.

In [ ]:
print(
    "Number of observations:",
    len(data)
)

print(
    "Start timestamp:",
    data["timestamp"].min()
)

print(
    "End timestamp:",
    data["timestamp"].max()
)

print(
    "Missing values:",
    data.isna().sum().sum()
)

In [ ]:
data[
    [
        "generator_power",
        "unit_speed",
        "tgb_temperature",
        "tgb_vibration",
        "ugb_temperature",
        "ugb_vibration"
    ]
].describe()

In [ ]:
data["month"] = (
    data["timestamp"]
    .dt.month
)

month_names = {
    6: "June",
    7: "July",
    8: "August",
    9: "September",
    10: "October",
    11: "November"
}

data["month_name"] = (
    data["month"]
    .map(month_names)
)

In [ ]:
data.groupby(
    "month_name"
).size()

## 5. Turbine Guide Bearing Vibration

The distribution of TGB vibration measurements is examined to understand its typical operating range.

In [ ]:
plt.figure(
    figsize=(9, 5)
)

plt.hist(
    data["tgb_vibration"],
    bins=20,
    edgecolor="black"
)

plt.xlabel(
    "TGB Vibration"
)

plt.ylabel(
    "Frequency"
)

plt.title(
    "Distribution of TGB Vibration"
)

plt.tight_layout()

plt.show()

In [ ]:
print(
    data["tgb_vibration"]
    .value_counts()
    .sort_index()
)

## 6. Upper Guide Bearing Vibration

The distribution of UGB vibration measurements is examined to understand its typical operating range.

In [ ]:
plt.figure(
    figsize=(9, 5)
)

plt.hist(
    data["ugb_vibration"],
    bins=20,
    edgecolor="black"
)

plt.xlabel(
    "UGB Vibration"
)

plt.ylabel(
    "Frequency"
)

plt.title(
    "Distribution of UGB Vibration"
)

plt.tight_layout()

plt.show()

In [ ]:
print(
    data["ugb_vibration"]
    .value_counts()
    .sort_index()
)

## 7. TGB and UGB Vibration Over Time

The vibration trends of both connected guide bearings are compared across the available operating period.

In [ ]:
plt.figure(
    figsize=(15, 5)
)

plt.plot(
    data["timestamp"],
    data["tgb_vibration"],
    label="TGB Vibration",
    linewidth=1
)

plt.plot(
    data["timestamp"],
    data["ugb_vibration"],
    label="UGB Vibration",
    linewidth=1
)

plt.xlabel(
    "Time"
)

plt.ylabel(
    "Vibration"
)

plt.title(
    "TGB and UGB Vibration Over Time"
)

plt.legend()

plt.tight_layout()

plt.show()

## 8. Relationship Between TGB and UGB Vibration

The relationship between the vibration measurements of the two connected guide bearings is examined using correlation analysis.

In [ ]:
vibration_correlation = (
    data[
        [
            "tgb_vibration",
            "ugb_vibration"
        ]
    ]
    .corr()
)

vibration_correlation

In [ ]:
tgb_ugb_corr = (
    vibration_correlation.loc[
        "tgb_vibration",
        "ugb_vibration"
    ]
)

print(
    f"TGB and UGB vibration correlation: "
    f"{tgb_ugb_corr:.4f}"
)

In [ ]:
plt.figure(
    figsize=(7, 6)
)

plt.scatter(
    data["tgb_vibration"],
    data["ugb_vibration"],
    alpha=0.25
)

plt.xlabel(
    "TGB Vibration"
)

plt.ylabel(
    "UGB Vibration"
)

plt.title(
    "Relationship Between TGB and UGB Vibration"
)

plt.tight_layout()

plt.show()

## 9. Bearing Temperature Behaviour

The temperature distributions of the Turbine Guide Bearing and Upper Guide Bearing are compared.

In [ ]:
plt.figure(
    figsize=(9, 5)
)

plt.hist(
    data["tgb_temperature"],
    bins=30,
    edgecolor="black"
)

plt.xlabel(
    "TGB Temperature"
)

plt.ylabel(
    "Frequency"
)

plt.title(
    "Distribution of TGB Temperature"
)

plt.tight_layout()

plt.show()

In [ ]:
plt.figure(
    figsize=(9, 5)
)

plt.hist(
    data["ugb_temperature"],
    bins=30,
    edgecolor="black"
)

plt.xlabel(
    "UGB Temperature"
)

plt.ylabel(
    "Frequency"
)

plt.title(
    "Distribution of UGB Temperature"
)

plt.tight_layout()

plt.show()

In [ ]:
plt.figure(
    figsize=(15, 5)
)

plt.plot(
    data["timestamp"],
    data["tgb_temperature"],
    label="TGB Temperature",
    linewidth=1
)

plt.plot(
    data["timestamp"],
    data["ugb_temperature"],
    label="UGB Temperature",
    linewidth=1
)

plt.xlabel(
    "Time"
)

plt.ylabel(
    "Temperature"
)

plt.title(
    "TGB and UGB Temperature Over Time"
)

plt.legend()

plt.tight_layout()

plt.show()

## 10. Correlation Analysis

Correlation analysis is performed among the bearing measurements and unit operating variables to identify relationships that may contribute to vibration forecasting.

In [ ]:
correlation_columns = [
    "generator_power",
    "unit_speed",
    "tgb_temperature",
    "tgb_vibration",
    "ugb_temperature",
    "ugb_vibration"
]

correlation_matrix = (
    data[
        correlation_columns
    ]
    .corr()
)

correlation_matrix.round(3)

In [ ]:
fig, ax = plt.subplots(
    figsize=(9, 7)
)

image = ax.imshow(
    correlation_matrix,
    aspect="auto"
)

ax.set_xticks(
    range(
        len(correlation_columns)
    )
)

ax.set_yticks(
    range(
        len(correlation_columns)
    )
)

ax.set_xticklabels(
    correlation_columns,
    rotation=45,
    ha="right"
)

ax.set_yticklabels(
    correlation_columns
)

for i in range(
    len(correlation_columns)
):

    for j in range(
        len(correlation_columns)
    ):

        ax.text(
            j,
            i,
            f"{correlation_matrix.iloc[i, j]:.2f}",
            ha="center",
            va="center"
        )

plt.title(
    "Correlation Matrix of HydroTwin Variables"
)

plt.colorbar(
    image
)

plt.tight_layout()

plt.show()

In [ ]:
tgb_correlations = (
    correlation_matrix[
        "tgb_vibration"
    ]
    .sort_values(
        ascending=False
    )
)

tgb_correlations

In [ ]:
ugb_correlations = (
    correlation_matrix[
        "ugb_vibration"
    ]
    .sort_values(
        ascending=False
    )
)

ugb_correlations

## 11. Monthly Vibration Behaviour

Average TGB and UGB vibration levels are compared across the available months to examine changes in component behaviour over time.

In [ ]:
monthly_vibration = (
    data
    .groupby("month_name")
    .agg(
        tgb_mean=(
            "tgb_vibration",
            "mean"
        ),
        tgb_max=(
            "tgb_vibration",
            "max"
        ),
        ugb_mean=(
            "ugb_vibration",
            "mean"
        ),
        ugb_max=(
            "ugb_vibration",
            "max"
        )
    )
)

monthly_vibration

In [ ]:
data["split"] = "Excluded"

data.loc[
    data["month"].isin(
        [6, 7, 8]
    ),
    "split"
] = "Training"

data.loc[
    data["month"] == 9,
    "split"
] = "Validation"

data.loc[
    data["month"] == 11,
    "split"
] = "Test"

In [ ]:
split_statistics = (
    data[
        data["split"] != "Excluded"
    ]
    .groupby("split")
    .agg(
        observations=(
            "timestamp",
            "size"
        ),

        tgb_mean=(
            "tgb_vibration",
            "mean"
        ),

        tgb_std=(
            "tgb_vibration",
            "std"
        ),

        ugb_mean=(
            "ugb_vibration",
            "mean"
        ),

        ugb_std=(
            "ugb_vibration",
            "std"
        )
    )
)

split_statistics

## 12. Save Exploratory Analysis Results

Key summary tables are saved for use in later model development and reporting.

In [ ]:
RESULTS_DIR = (
    PROJECT_ROOT /
    "results"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [ ]:
correlation_matrix.to_csv(
    RESULTS_DIR /
    "feature_correlation_matrix.csv"
)

In [ ]:
monthly_vibration.to_csv(
    RESULTS_DIR /
    "monthly_vibration_summary.csv"
)

In [ ]:
split_statistics.to_csv(
    RESULTS_DIR /
    "dataset_split_statistics.csv"
)

In [ ]:
print(
    "EDA results saved successfully."
)

## 13. EDA Summary

The exploratory analysis examined the behaviour of the two connected guide bearing components and their relationships with the unit operating variables.

The cleaned dataset contains stable operating measurements for both TGB and UGB. Their vibration and temperature measurements vary across the operating period, while generator power reflects changing unit load conditions.

The observed relationship between TGB and UGB supports their representation as connected components within the HydroTwin digital twin prototype.

The next stage develops the Linear Regression baseline model using the prepared sequences.